# 15: Census Demographics Integration — Quick Start

> **Comprehensive coverage** of Census boundaries, GEOID utilities, crosswalks, and time-series
> analysis is in **[04_Spatial_Data_Census_Boundaries.ipynb](04_Spatial_Data_Census_Boundaries.ipynb)**.
> This notebook is a **quick-start guide** for the convenience functions only.

This notebook demonstrates the **convenience functions** for fetching Census demographic data:

1. **Variable Groups** — Predefined sets of common Census variables
2. **Quick Fetches** — `get_population()`, `get_income_data()`, etc.
3. **Django Cache Backend** — Using Django's cache framework with CensusAPIClient
4. **PL 94-171 Redistricting Data** — Block-level redistricting data

For `CensusAPIClient`, `get_census_data_with_geometry()`, GEOID utilities,
crosswalks, and time-series analysis, see NB04.

# 15: Census Demographics Integration — Quick Start

> **Comprehensive coverage** of Census boundaries, GEOID utilities, crosswalks, and time-series
> analysis is in **[04_Spatial_Data_Census_Boundaries.ipynb](04_Spatial_Data_Census_Boundaries.ipynb)**.
> This notebook is a **quick-start guide** for the convenience functions only.

This notebook demonstrates the **convenience functions** for fetching Census demographic data:

1. **Variable Groups** — Predefined sets of common Census variables
2. **Quick Fetches** — `get_population()`, `get_income_data()`, etc.

For `CensusAPIClient`, `get_census_data_with_geometry()`, GEOID utilities,
crosswalks, and time-series analysis, see NB04.

In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

# Quick-start convenience functions
from siege_utilities.geo.census_api_client import (
    get_population,
    get_income_data,
    get_demographics,
    get_education_data,
    get_housing_data,
    VARIABLE_GROUPS
)

su.log_info("Imports successful!")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_15"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")

## 1. Understanding Variable Groups

The Census Bureau uses standardized variable codes (e.g., `B01001_001E` for total population).
siege_utilities provides predefined groups for common analyses.

In [ ]:
# List all predefined variable groups
su.log_info("Predefined Variable Groups")
su.log_info("=" * 60)

for group_name, variables in VARIABLE_GROUPS.items():
    su.log_info(f"{group_name}:")
    for var in variables[:5]:  # Show first 5 variables
        su.log_info(f"  - {var}")
    if len(variables) > 5:
        su.log_info(f"  ... and {len(variables) - 5} more")

## 2. Quick Demographic Fetches

Use convenience functions for common data needs.

In [ ]:
# Fetch population for all California counties
try:
    ca_pop = get_population(
        state='California',
        geography='county',
        year=2022
    )
    
    su.log_info(f"Retrieved population for {len(ca_pop)} California counties")
    su.log_info(f"Columns: {list(ca_pop.columns)}")
    
    # Rename for clarity and show top counties
    ca_pop = ca_pop.rename(columns={'B01001_001E': 'total_population'})
    su.log_info("Top 10 Most Populous California Counties:")
    display(ca_pop.nlargest(10, 'total_population')[['NAME', 'total_population']])
    
except Exception as e:
    su.log_error(f"Error: {e}")
    su.log_info("Tip: Set CENSUS_API_KEY environment variable for reliable access")

In [ ]:
# Fetch income data for Texas counties
try:
    tx_income = get_income_data(
        state='TX',
        geography='county',
        year=2022
    )
    
    # Show median household income (B19013_001E)
    tx_income = tx_income.rename(columns={'B19013_001E': 'median_hh_income'})
    
    su.log_info("Texas Counties - Highest Median Household Income:")
    display(tx_income.nlargest(5, 'median_hh_income')[['NAME', 'median_hh_income']])
    
    su.log_info("Texas Counties - Lowest Median Household Income:")
    # Filter out null values before getting smallest
    valid = tx_income[tx_income['median_hh_income'] > 0]
    display(valid.nsmallest(5, 'median_hh_income')[['NAME', 'median_hh_income']])
    
except Exception as e:
    su.log_error(f"Error: {e}")

## For More: See NB04

The following topics are covered in detail in **[04_Spatial_Data_Census_Boundaries.ipynb](04_Spatial_Data_Census_Boundaries.ipynb)**:

- **CensusAPIClient** — Full control over Census API queries (`fetch_data()`, `get_variable_metadata()`)
- **Combined Geometry + Demographics** — `get_census_data_with_geometry()` for one-call GeoDataFrames
- **Census Tract Analysis** — Finer-grained analysis below county level
- **Manual Shape + Data Joins** — `join_demographics_to_shapes()` for custom joins
- **GEOID Utilities** — `construct_geoid()`, `parse_geoid()`, `normalize_geoid()`, `validate_geoid()`
- **Intelligent Dataset Selection** — `CensusDataSelector` for dataset recommendations
- **Boundary Crosswalks** — `CrosswalkClient` for vintage-to-vintage mappings
- **Time-Series Analysis** — `classify_trends()`, `identify_outliers()`, dual-mode Spark support

## 3. Django Cache Backend

The `CensusAPIClient` supports Django's cache framework as an alternative to the default
parquet file cache. This is useful in Django projects where you want shared cache management.

```python
from siege_utilities.geo.census_api_client import CensusAPIClient

# Use Django's cache framework (requires CACHES in settings.py)
client = CensusAPIClient(cache_backend='django')

# Fetches are cached in Django's default cache (Redis, Memcached, etc.)
df = client.fetch_data(
    variables='income',
    year=2022,
    dataset='acs5',
    geography='county',
    state_fips='06',
)
# Subsequent calls with same params return from cache
```

## Summary

### Quick Convenience Functions

| Function | Description |
|----------|-------------|
| `get_population()` | Total population counts |
| `get_demographics()` | Basic demographic variables |
| `get_income_data()` | Income-related variables |
| `get_education_data()` | Educational attainment |
| `get_housing_data()` | Housing characteristics |

### Common Variable Codes

| Code | Description |
|------|-------------|
| B01001_001E | Total Population |
| B19013_001E | Median Household Income |
| B25077_001E | Median Home Value |
| B15003_022E | Bachelor's Degree |
| B17001_002E | Population Below Poverty |

### See Also

- **[04_Spatial_Data_Census_Boundaries.ipynb](04_Spatial_Data_Census_Boundaries.ipynb)** — Boundaries, CensusAPIClient, GEOIDs, crosswalks, time-series
- **[20_Multi_Source_Spatial_Tabulation.ipynb](20_Multi_Source_Spatial_Tabulation.ipynb)** — Cross-source tabulation (Census × NCES × NLRB × RDH)
- [Census API Key Signup](https://api.census.gov/data/key_signup.html)